# Encoding Sensitivity Check — LabelEncoder vs. One-Hot (Adult dataset)

**Status: FOR DISCUSSION, not yet adopted.** Item 3a on tomorrow's meeting
list — deciding whether to switch to One-Hot encoding (Reviewer 3, comment
8) or argue in text that the choice doesn't affect conclusions.

Compares K-means cluster assignments under the two encodings via Adjusted
Rand Index (ARI): high ARI → encoding choice doesn't matter much; low ARI →
it does, and simply arguing it away in text would not be defensible.

**Set `ADULT_PATH` below to your real Adult data file before drawing any
conclusion.** Without it, this notebook falls back to a small synthetic
mock (same columns, random values) purely so the code itself can be
verified to run — the resulting ARI number is NOT meaningful (random data
has no real cluster structure, so any encoding will look "sensitive"). Only
run against the real file for a real answer.

## Cell 1: Imports + config

In [3]:
"""
Encoding sensitivity check: LabelEncoder vs One-Hot, on the Adult dataset.
NOT part of the main pipeline -- a standalone diagnostic to inform the
Reviewer 3, comment 8 discussion (LabelEncoder imposes arbitrary ordinal
distance on nominal categories). Status: FOR DISCUSSION, not yet adopted.

If the real Adult data file is not found at ADULT_PATH, this script falls
back to a small synthetic mock with the same column structure, purely so
the script itself can be verified to run end-to-end. Point ADULT_PATH at
your real file to get the real answer.
"""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score

ADULT_PATH = "C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/adult/adult.data"   # point this at your real file

## Cell 2: Column names + data loader (falls back to synthetic mock if real file not found)

In [5]:
ADULT_COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]


def load_adult(path=ADULT_PATH):
    try:
        adult = pd.read_csv(path, names=ADULT_COLUMNS, header=None)
        adult = adult.replace(' ?', np.nan).dropna()
        print(f"Loaded real Adult data from {path}: {adult.shape}")
        return adult
    except FileNotFoundError:
        print(f"[DEMO MODE] Could not find {path} -- using a small synthetic "
              f"mock with the same column structure so this script can be "
              f"verified to run. Point ADULT_PATH at your real file for the "
              f"real answer.")
        rng = np.random.RandomState(0)
        n = 1000
        adult = pd.DataFrame({
            'age': rng.randint(18, 70, n),
            'workclass': rng.choice(['Private', 'Self-emp-not-inc', 'Local-gov',
                                      'State-gov', 'Federal-gov'], n),
            'fnlwgt': rng.randint(10000, 500000, n),
            'education': rng.choice(['Bachelors', 'HS-grad', 'Masters', 'Some-college',
                                      'Doctorate', '11th', '9th'], n),
            'education-num': rng.randint(1, 16, n),
            'marital-status': rng.choice(['Married-civ-spouse', 'Never-married',
                                           'Divorced', 'Separated', 'Widowed'], n),
            'occupation': rng.choice(['Tech-support', 'Sales', 'Exec-managerial',
                                       'Prof-specialty', 'Craft-repair', 'Other-service'], n),
            'relationship': rng.choice(['Husband', 'Not-in-family', 'Wife',
                                         'Own-child', 'Unmarried'], n),
            'race': rng.choice(['White', 'Black', 'Asian-Pac-Islander'], n),
            'sex': rng.choice(['Male', 'Female'], n),
            'capital-gain': rng.randint(0, 10000, n),
            'capital-loss': rng.randint(0, 2000, n),
            'hours-per-week': rng.randint(1, 80, n),
            'native-country': rng.choice(['United-States', 'Mexico', 'Philippines',
                                           'Germany', 'India'], n),
            'income': rng.choice(['<=50K', '>50K'], n),
        })
        return adult

## Cell 3: The two preprocessing pipelines

In [7]:
def preprocess_labelencoder(df, feature_cols):
    """Matches the existing preprocess_dataset() in the main pipeline."""
    X_df = df[feature_cols].copy()
    for col in X_df.select_dtypes(include=['object']).columns:
        X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))
    X = X_df.values.astype(float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return StandardScaler().fit_transform(X)


def preprocess_onehot(df, feature_cols):
    """One-hot encode categorical columns; numeric columns pass through."""
    X_df = df[feature_cols].copy()
    cat_cols = X_df.select_dtypes(include=['object']).columns.tolist()
    num_cols = [c for c in feature_cols if c not in cat_cols]

    num_part = X_df[num_cols].values.astype(float) if num_cols else np.zeros((len(X_df), 0))
    if cat_cols:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        cat_part = ohe.fit_transform(X_df[cat_cols].astype(str))
    else:
        cat_part = np.zeros((len(X_df), 0))

    X = np.hstack([num_part, cat_part])
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return StandardScaler().fit_transform(X), cat_part.shape[1] + num_part.shape[1]

## Cell 4: Run the comparison

In [9]:
def run_sensitivity_check(k=5, n_seeds=10):
    adult = load_adult()
    feature_cols = [c for c in adult.columns if c not in ['income', 'sex', 'race']]

    X_le = preprocess_labelencoder(adult, feature_cols)
    X_oh, n_dims_oh = preprocess_onehot(adult, feature_cols)

    print(f"\nFeature dimensionality: LabelEncoder={X_le.shape[1]}  One-Hot={n_dims_oh}"
          f"  (expansion factor: {n_dims_oh / X_le.shape[1]:.1f}x)")

    aris, sils_le, sils_oh = [], [], []
    for seed in range(n_seeds):
        labels_le = KMeans(n_clusters=k, random_state=seed, n_init=10).fit_predict(X_le)
        labels_oh = KMeans(n_clusters=k, random_state=seed, n_init=10).fit_predict(X_oh)
        aris.append(adjusted_rand_score(labels_le, labels_oh))
        sils_le.append(silhouette_score(X_le, labels_le))
        sils_oh.append(silhouette_score(X_oh, labels_oh))

    print(f"\n{'='*60}")
    print(f"ENCODING SENSITIVITY CHECK (k={k}, {n_seeds} seeds)")
    print(f"{'='*60}")
    print(f"ARI (LabelEncoder clustering vs. One-Hot clustering):")
    print(f"  mean={np.mean(aris):.3f}  std={np.std(aris):.3f}  "
          f"range=[{np.min(aris):.3f}, {np.max(aris):.3f}]")
    print(f"\nSilhouette under LabelEncoder: mean={np.mean(sils_le):.3f}  std={np.std(sils_le):.3f}")
    print(f"Silhouette under One-Hot:      mean={np.mean(sils_oh):.3f}  std={np.std(sils_oh):.3f}")


    return {
        'ari_mean': float(np.mean(aris)), 'ari_std': float(np.std(aris)),
        'ari_values': aris,
        'silhouette_labelencoder': float(np.mean(sils_le)),
        'silhouette_onehot': float(np.mean(sils_oh)),
        'n_dims_labelencoder': int(X_le.shape[1]),
        'n_dims_onehot': int(n_dims_oh),
    }

## Cell 5: Execute

In [11]:
results = run_sensitivity_check(k=5, n_seeds=10)

Loaded real Adult data from C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/adult/adult.data: (30162, 15)

Feature dimensionality: LabelEncoder=12  One-Hot=97  (expansion factor: 8.1x)

ENCODING SENSITIVITY CHECK (k=5, 10 seeds)
ARI (LabelEncoder clustering vs. One-Hot clustering):
  mean=0.247  std=0.035  range=[0.200, 0.297]

Silhouette under LabelEncoder: mean=0.152  std=0.012
Silhouette under One-Hot:      mean=0.062  std=0.010
